# ANTARES hybrid downloader — add 100k more unique loci with random seeds

This notebook continues from the existing clean v10 dataset in:

```text
antares_data_hybrid/
```

The existing dataset contains **9,999 unique loci**.  
The goal of this notebook is to add **100,000 more unique retained loci**, for a target of:

```text
109,999 total unique loci on disk
```

## Strategy

- Use **ANTARES API v2** for locus search and alert retrieval.
- Use **ANTARES client v1** only for detailed Gaia `catalog_objects`.
- Keep the same Lantern selection:
  - tag = `lantern_xgboost_t2.0.7_c0.95`
  - `num_tagged_alerts > 2`
  - `max_score > 0.9691`
- Do **not** use `newest_alert_observation_time` in the v2 query.
- Wrap the locus query in Elasticsearch `function_score` with `random_score`.
- Use deterministic seeds `1, 2, 3, ...`.
- ANTARES v2 can expose only about one ~10k result window for a given query/seed, so after one seed is exhausted, move to the next seed.
- Existing and newly encountered `locus_id` values are deduplicated.
- Existing and newly encountered `alert_id` values are deduplicated.
- Keep only LSST alerts with `MJD >= 61187.0` = **2026-05-27 00:00 UTC**.
- Discard a locus if it has no retained LSST alert.
- Continue writing new Parquet parts into the same existing folders.
- Keep independent locus and alert batches.
- Save a small seed checkpoint so interrupted runs can resume safely.

The random-seed behavior was tested directly against ANTARES v2:

```text
same seed reproducible: True
different seeds differ: True
```

The existing 9,999 loci are never deleted or rewritten.


## 1. Imports

In [ ]:
import os

# Force the ANTARES client to use API v1 for get_by_id/catalog_objects.
os.environ["ANTARES_API_BASE_URL"] = "https://api.antares.noirlab.edu/v1/"

import json
import time
import requests
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from pathlib import Path
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed

from antares_client.search import get_by_id


## 2. Settings and v2 query

In [ ]:
V2_BASE_URL = "https://api.antares.noirlab.edu/v2/"

outdir = Path("antares_data_hybrid")
locus_dir = outdir / "loci"
alert_dir = outdir / "alerts"

locus_dir.mkdir(parents=True, exist_ok=True)
alert_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Target
# ------------------------------------------------------------------
# Existing clean v10 dataset: 9,999 unique loci
# Desired extension:          100,000 additional unique loci
# Final target:               109,999 unique loci total
TARGET_TOTAL_LOCI = 60_000 #109_999

# Random-score seeds are deterministic.
SEED_START = 1

# Resume checkpoint.
# This stores the seed that should be attempted next/currently.
SEED_STATE_FILE = outdir / "random_seed_state.json"

# Parallel network requests
MAX_WORKERS = 8
FETCH_BATCH_SIZE = 32

# API page size
API_LIMIT = 100

# Independent disk batching
LOCUS_BATCH_SIZE = 5_000
ALERT_BATCH_SIZE = 100_000

# Network retry settings
N_RETRY = 3
REQUEST_TIMEOUT = 60

# Keep alerts from 2026-05-27 00:00 UTC onward
MJD_MIN = 61187.0

TAG_NAME = "lantern_xgboost_t2.0.7_c0.95"


# ------------------------------------------------------------------
# Base Lantern selection
# ------------------------------------------------------------------
BASE_FILTER = [
    {
        "range": {
            "properties.lantern_xgboost_t2.0.7_c0.95_num_tagged_alerts": {
                "gt": 2
            }
        }
    },
    {
        "range": {
            "properties.lantern_xgboost_t2.0.7_c0.95_max_score": {
                "gt": 0.9691
            }
        }
    },
    {
        "terms": {
            "tags": [TAG_NAME]
        }
    },
]


def make_random_query(seed):
    """
    Same Lantern selection as v10, but randomize the locus ordering/sample
    reproducibly with an Elasticsearch random_score seed.
    """
    return {
        "query": {
            "function_score": {
                "query": {
                    "bool": {
                        "filter": BASE_FILTER
                    }
                },
                "random_score": {
                    "seed": int(seed),
                    "field": "_seq_no"
                },
                "boost_mode": "replace",
            }
        }
    }


make_random_query(SEED_START)


## 3. Parquet schemas

In [ ]:
locus_schema = pa.schema([
    ("locus_id", pa.string()),
    ("ra", pa.float64()),
    ("dec", pa.float64()),

    ("tags", pa.list_(pa.string())),
    ("catalogs", pa.list_(pa.string())),

    ("num_tagged_alerts", pa.int64()),
    ("max_score", pa.float64()),
    ("ztf_object_id", pa.string()),
    ("survey_lsst", pa.string()),

    # Gaia DR3 variability
    ("gaia_variability_class", pa.list_(pa.string())),

    # Gaia DR3 source
    ("gaia_parallax_over_error", pa.list_(pa.float64())),
    ("gaia_pmdec", pa.list_(pa.float64())),
    ("gaia_pmdec_error", pa.list_(pa.float64())),
    ("gaia_pmra", pa.list_(pa.float64())),
    ("gaia_pmra_error", pa.list_(pa.float64())),
    ("gaia_in_qso_candidates", pa.list_(pa.bool_())),

    ("gaia_phot_g_mean_mag", pa.list_(pa.float64())),
    ("gaia_phot_g_mean_flux_over_error", pa.list_(pa.float64())),

    # Gaia colors
    ("gaia_bp_g", pa.list_(pa.float64())),   # BP - G
    ("gaia_g_rp", pa.list_(pa.float64())),   # G - RP

    # Matched catalog flag
    ("milliquas_match", pa.bool_()),
])


alert_schema = pa.schema([
    ("locus_id", pa.string()),
    ("alert_id", pa.string()),
    ("mjd", pa.float64()),

    # Keep all LSST alert properties
    ("alert_properties", pa.string()),
])


## 4. Helpers

In [ ]:
def compact_json(x):
    if x is None:
        return None

    return json.dumps(
        x,
        separators=(",", ":"),
        default=str,
    )


def to_float(x):
    return None if x is None else float(x)


def to_int(x):
    return None if x is None else int(x)


def to_bool(x):
    if x is None:
        return None

    if isinstance(x, bool):
        return x

    if isinstance(x, str):
        x = x.strip().lower()

        if x == "true":
            return True

        if x == "false":
            return False

    return bool(x)


def get_json(url, params=None):
    """GET JSON from API v2 with simple retries."""

    for attempt in range(N_RETRY):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError):

            if attempt == N_RETRY - 1:
                raise

            time.sleep(2 ** attempt)


def get_all_v2_data(url, params=None):
    """Retrieve all pages from a paginated API-v2 endpoint."""

    rows = []

    while url:

        response = get_json(
            url,
            params=params,
        )

        rows.extend(
            response.get("data", [])
        )

        next_url = (
            response
            .get("links", {})
            .get("next")
        )

        url = (
            urljoin(url, next_url)
            if next_url
            else None
        )

        # The next URL already contains pagination information.
        params = None

    return rows

def iter_v2_loci(seed):
    """Iterate over one paginated API-v2 random-score locus query."""

    query = make_random_query(seed)

    url = V2_BASE_URL + "loci"

    params = {
        "limit": API_LIMIT,
        "elasticsearch_query": json.dumps(query),
    }

    while url:

        response = get_json(
            url,
            params=params,
        )

        for item in response.get("data", []):
            yield item.get("locus", item)

        next_url = (
            response
            .get("links", {})
            .get("next")
        )

        url = (
            urljoin(url, next_url)
            if next_url
            else None
        )

        # The next URL already contains pagination information.
        params = None


In [ ]:
def normalize_catalogs(catalogs):
    names = []

    for catalog in catalogs or []:

        if isinstance(catalog, str):
            names.append(catalog)

        elif isinstance(catalog, dict):

            name = (
                catalog.get("name")
                or catalog.get("catalog_name")
            )

            if name is not None:
                names.append(str(name))

    return names


def get_catalog_objects_v1(locus_id):
    """Retrieve matched catalog objects using ANTARES client v1."""

    for attempt in range(N_RETRY):

        try:
            locus_v1 = get_by_id(locus_id)

            if locus_v1 is None:
                raise RuntimeError(
                    f"Could not retrieve {locus_id} through API v1"
                )

            return locus_v1.catalog_objects or {}

        except Exception:

            if attempt == N_RETRY - 1:
                raise

            time.sleep(2 ** attempt)


def load_existing_ids(folder, column):
    """Load IDs already saved in previous runs."""

    files = list(folder.glob("*.parquet"))

    if not files:
        return set()

    table = ds.dataset(
        folder,
        format="parquet",
    ).to_table(
        columns=[column]
    )

    return set(
        table[column].to_pylist()
    )


def next_part_number(folder, prefix):
    files = list(
        folder.glob(f"{prefix}_*.parquet")
    )

    if not files:
        return 0

    return (
        max(
            int(f.stem.split("_")[-1])
            for f in files
        )
        + 1
    )


def write_parquet(rows, schema, filename):
    if not rows:
        return

    table = pa.Table.from_pylist(
        rows,
        schema=schema,
    )

    pq.write_table(
        table,
        filename,
        compression="zstd",
    )


## 5. Process one locus

In [ ]:
def process_locus(locus):

    locus_id = str(locus["locus_id"])
    props = locus.get("properties", {}) or {}

    catalogs = normalize_catalogs(
        locus.get("catalogs", [])
    )

    catalog_set = set(catalogs)

    # Presence in the matched catalog list is sufficient.
    milliquas_match = any(
        "milliquas" in name.lower()
        for name in catalogs
    )

    survey = props.get("survey", {})

    survey_lsst = (
        survey.get("lsst")
        if isinstance(survey, dict)
        else None
    )


    # ========================================================
    # Alerts: API v2
    # ========================================================

    alert_data = get_all_v2_data(
        V2_BASE_URL + f"loci/{locus_id}/alerts",
        params={
            "limit": API_LIMIT,
            "elasticsearch_query": json.dumps({}),
        },
    )

    alert_rows = []
    seen_alert_ids_local = set()

    for item in alert_data:

        alert = item.get("alert", item)

        # API v2 alert identifier
        alert_id = alert.get("id")

        if alert_id is None:
            continue

        alert_id = str(alert_id)

        # Keep only LSST alerts.
        if not alert_id.startswith("lsst:"):
            continue

        # Deduplicate within the locus.
        if alert_id in seen_alert_ids_local:
            continue

        seen_alert_ids_local.add(alert_id)

        mjd = to_float(
            alert.get("mjd")
        )

        # Discard alerts before 2026-05-27.
        if mjd is None or mjd < MJD_MIN:
            continue

        alert_rows.append({
            "locus_id": locus_id,
            "alert_id": alert_id,
            "mjd": mjd,

            # Keep all LSST alert properties.
            "alert_properties": compact_json(
                alert.get("properties", {})
            ),
        })


    # Save only loci with at least one retained LSST alert.
    if not alert_rows:
        return None, []


    # ========================================================
    # Matched catalog details: ANTARES client v1
    # ========================================================

    gaia_variability_class = []

    gaia_parallax_over_error = []
    gaia_pmdec = []
    gaia_pmdec_error = []
    gaia_pmra = []
    gaia_pmra_error = []
    gaia_in_qso_candidates = []

    gaia_phot_g_mean_mag = []
    gaia_phot_g_mean_flux_over_error = []

    gaia_bp_g = []
    gaia_g_rp = []


    # Only query detailed catalog objects when Gaia is matched.
    need_gaia = (
        "gaia_dr3_variability" in catalog_set
        or "gaia_dr3_gaia_source" in catalog_set
    )

    if need_gaia:

        catalog_objects = get_catalog_objects_v1(
            locus_id
        )


        # Gaia DR3 variability
        for obj in catalog_objects.get(
            "gaia_dr3_variability",
            [],
        ):

            value = obj.get("class")

            gaia_variability_class.append(
                str(value)
                if value is not None
                else None
            )


        # Gaia DR3 source
        for obj in catalog_objects.get(
            "gaia_dr3_gaia_source",
            [],
        ):

            gaia_parallax_over_error.append(
                to_float(
                    obj.get("parallax_over_error")
                )
            )

            gaia_pmdec.append(
                to_float(
                    obj.get("pmdec")
                )
            )

            gaia_pmdec_error.append(
                to_float(
                    obj.get("pmdec_error")
                )
            )

            gaia_pmra.append(
                to_float(
                    obj.get("pmra")
                )
            )

            gaia_pmra_error.append(
                to_float(
                    obj.get("pmra_error")
                )
            )

            gaia_in_qso_candidates.append(
                to_bool(
                    obj.get("in_qso_candidates")
                )
            )

            gaia_phot_g_mean_mag.append(
                to_float(
                    obj.get("phot_g_mean_mag")
                )
            )

            gaia_phot_g_mean_flux_over_error.append(
                to_float(
                    obj.get(
                        "phot_g_mean_flux_over_error"
                    )
                )
            )

            gaia_bp_g.append(
                to_float(
                    obj.get("bp_g")
                )
            )

            gaia_g_rp.append(
                to_float(
                    obj.get("g_rp")
                )
            )


    # ========================================================
    # Locus row
    # ========================================================

    locus_row = {
        "locus_id": locus_id,

        "ra": to_float(
            locus.get("ra")
        ),

        "dec": to_float(
            locus.get("dec")
        ),

        "tags": [
            str(x)
            for x in locus.get("tags", [])
        ],

        "catalogs": catalogs,

        "num_tagged_alerts": to_int(
            props.get(
                "lantern_xgboost_t2.0.7_c0.95_num_tagged_alerts"
            )
        ),

        "max_score": to_float(
            props.get(
                "lantern_xgboost_t2.0.7_c0.95_max_score"
            )
        ),

        "ztf_object_id": (
            str(props["ztf_object_id"])
            if props.get("ztf_object_id") is not None
            else None
        ),

        "survey_lsst": compact_json(
            survey_lsst
        ),

        "gaia_variability_class":
            gaia_variability_class,

        "gaia_parallax_over_error":
            gaia_parallax_over_error,

        "gaia_pmdec":
            gaia_pmdec,

        "gaia_pmdec_error":
            gaia_pmdec_error,

        "gaia_pmra":
            gaia_pmra,

        "gaia_pmra_error":
            gaia_pmra_error,

        "gaia_in_qso_candidates":
            gaia_in_qso_candidates,

        "gaia_phot_g_mean_mag":
            gaia_phot_g_mean_mag,

        "gaia_phot_g_mean_flux_over_error":
            gaia_phot_g_mean_flux_over_error,

        "gaia_bp_g":
            gaia_bp_g,

        "gaia_g_rp":
            gaia_g_rp,

        "milliquas_match":
            milliquas_match,
    }

    return locus_row, alert_rows


## API-v2 alert format check

Optional quick check before the full run. This confirms that the v2 alert endpoint returns `id`, `mjd`, and `properties` for a queried locus.

In [ ]:
test_locus_id = "ANT2026lvsrmjx07t0w"

test_alerts = get_all_v2_data(
    V2_BASE_URL + f"loci/{test_locus_id}/alerts",
    params={
        "limit": 2,
        "elasticsearch_query": json.dumps({}),
    },
)

print("Test locus:", test_locus_id)

if test_alerts:
    print("Alert keys:", test_alerts[0].keys())
    print("Alert id:", test_alerts[0].get("id"))
    print("MJD:", test_alerts[0].get("mjd"))
else:
    print("No alerts returned for this test locus.")


## 6. Download until 109,999 total unique loci new unique loci

In [ ]:
# IDs already on disk from previous runs.
existing_locus_ids = load_existing_ids(
    locus_dir,
    "locus_id",
)

existing_alert_ids = load_existing_ids(
    alert_dir,
    "alert_id",
)

initial_locus_count = len(existing_locus_ids)
initial_alert_count = len(existing_alert_ids)

remaining_at_start = max(
    0,
    TARGET_TOTAL_LOCI - initial_locus_count,
)

print(
    f"Existing unique loci:   "
    f"{initial_locus_count:,}"
)

print(
    f"Existing unique alerts: "
    f"{initial_alert_count:,}"
)

print(
    f"Target total loci:      "
    f"{TARGET_TOTAL_LOCI:,}"
)

print(
    f"Additional loci needed: "
    f"{remaining_at_start:,}"
)


# Continue locus and alert part numbers independently.
locus_part = next_part_number(
    locus_dir,
    "loci",
)

alert_part = next_part_number(
    alert_dir,
    "alerts",
)


locus_rows = []
alert_rows = []

# Tracks raw loci already encountered in this execution, including
# loci that later fail the recent-alert cut. This avoids repeated work
# when different random seeds overlap.
seen_locus_ids_this_run = set()

n_loci_saved = 0
n_alerts_saved = 0

n_duplicate_loci = 0
n_duplicate_alerts = 0
n_no_recent_alerts = 0
n_failed = 0
n_raw_loci_seen = 0


def flush_alerts():
    global alert_rows
    global alert_part

    if not alert_rows:
        return

    filename = (
        alert_dir
        / f"alerts_{alert_part:05d}.parquet"
    )

    write_parquet(
        alert_rows,
        alert_schema,
        filename,
    )

    print(
        f"Saved alert part {alert_part:05d}: "
        f"{len(alert_rows):,} alerts"
    )

    alert_rows = []
    alert_part += 1


def flush_loci():
    global locus_rows
    global locus_part

    if not locus_rows:
        return

    filename = (
        locus_dir
        / f"loci_{locus_part:05d}.parquet"
    )

    write_parquet(
        locus_rows,
        locus_schema,
        filename,
    )

    print(
        f"Saved locus part {locus_part:05d}: "
        f"{len(locus_rows):,} loci"
    )

    locus_rows = []
    locus_part += 1


def load_seed_state():
    """
    Return the seed to use.

    If no checkpoint exists, start from SEED_START.
    """
    if not SEED_STATE_FILE.exists():
        return SEED_START

    try:
        with open(SEED_STATE_FILE, "r") as f:
            state = json.load(f)

        return int(
            state.get(
                "next_seed",
                SEED_START,
            )
        )

    except Exception:
        # Do not let a malformed checkpoint destroy the download.
        return SEED_START


def save_seed_state(next_seed):
    """
    Save the seed checkpoint atomically.

    This is called only after all buffered data from a fully exhausted
    seed have been flushed to disk.
    """
    tmp = SEED_STATE_FILE.with_suffix(".tmp")

    with open(tmp, "w") as f:
        json.dump(
            {
                "next_seed": int(next_seed),
            },
            f,
            indent=2,
        )

    tmp.replace(SEED_STATE_FILE)


In [ ]:
%%time

seed = load_seed_state()

print()
print(
    f"Starting random-score seed: {seed}"
)
print()


try:

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        # ============================================================
        # Continue until the TOTAL number of unique loci on disk
        # reaches TARGET_TOTAL_LOCI.
        # ============================================================

        while (
            len(existing_locus_ids)
            < TARGET_TOTAL_LOCI
        ):

            print()
            print(
                f"========== Seed {seed} =========="
            )

            locus_iterator = iter_v2_loci(
                seed
            )

            seed_exhausted = False

            seed_raw_start = n_raw_loci_seen
            seed_saved_start = n_loci_saved
            seed_dup_start = n_duplicate_loci
            seed_no_recent_start = n_no_recent_alerts
            seed_failed_start = n_failed


            # ========================================================
            # Consume this seed until:
            #   1) the seed's API result stream is exhausted, or
            #   2) the total target is reached.
            # ========================================================

            while (
                len(existing_locus_ids)
                < TARGET_TOTAL_LOCI
            ):

                batch = []


                # ----------------------------------------------------
                # Select a batch not already saved/seen
                # ----------------------------------------------------

                while (
                    len(batch) < FETCH_BATCH_SIZE
                    and (
                        len(existing_locus_ids)
                        + len(batch)
                        < TARGET_TOTAL_LOCI
                    )
                ):

                    try:
                        locus = next(
                            locus_iterator
                        )

                    except StopIteration:
                        seed_exhausted = True
                        break


                    n_raw_loci_seen += 1

                    locus_id = str(
                        locus["locus_id"]
                    )


                    if (
                        locus_id in existing_locus_ids
                        or locus_id in seen_locus_ids_this_run
                    ):

                        n_duplicate_loci += 1
                        continue


                    seen_locus_ids_this_run.add(
                        locus_id
                    )

                    batch.append(
                        locus
                    )


                # No remaining usable loci from this seed.
                if not batch:
                    break


                # ----------------------------------------------------
                # Process loci in parallel
                # ----------------------------------------------------

                futures = {
                    executor.submit(
                        process_locus,
                        locus,
                    ): str(locus["locus_id"])

                    for locus in batch
                }


                for future in as_completed(
                    futures
                ):

                    locus_id = futures[
                        future
                    ]


                    try:

                        (
                            locus_row,
                            alert_rows_local,
                        ) = future.result()


                    except Exception as e:

                        n_failed += 1

                        print(
                            f"FAILED {locus_id}: {e}"
                        )

                        continue


                    # No retained LSST alert on/after 2026-05-27.
                    if locus_row is None:

                        n_no_recent_alerts += 1
                        continue


                    # ------------------------------------------------
                    # Global alert deduplication
                    # ------------------------------------------------

                    new_alert_rows = []

                    for row in alert_rows_local:

                        alert_id = row[
                            "alert_id"
                        ]

                        if (
                            alert_id
                            in existing_alert_ids
                        ):

                            n_duplicate_alerts += 1
                            continue


                        existing_alert_ids.add(
                            alert_id
                        )

                        new_alert_rows.append(
                            row
                        )


                    # ------------------------------------------------
                    # Add this locus and its alerts
                    # ------------------------------------------------

                    locus_rows.append(
                        locus_row
                    )

                    alert_rows.extend(
                        new_alert_rows
                    )

                    existing_locus_ids.add(
                        locus_id
                    )

                    n_loci_saved += 1

                    n_alerts_saved += len(
                        new_alert_rows
                    )


                    # ------------------------------------------------
                    # Flush alerts independently
                    # ------------------------------------------------

                    if (
                        len(alert_rows)
                        >= ALERT_BATCH_SIZE
                    ):

                        flush_alerts()


                    # ------------------------------------------------
                    # Flush loci independently
                    # ------------------------------------------------

                    if (
                        len(locus_rows)
                        >= LOCUS_BATCH_SIZE
                    ):

                        # Save pending alerts first so saved loci
                        # cannot be left without their alerts.
                        flush_alerts()

                        flush_loci()


                # ----------------------------------------------------
                # Progress
                # ----------------------------------------------------

                print(
                    f"Seed {seed} | "
                    f"total loci: "
                    f"{len(existing_locus_ids):,}"
                    f"/{TARGET_TOTAL_LOCI:,} | "
                    f"new this run: "
                    f"{n_loci_saved:,} | "
                    f"new alerts: "
                    f"{n_alerts_saved:,} | "
                    f"duplicate loci: "
                    f"{n_duplicate_loci:,} | "
                    f"no recent alerts: "
                    f"{n_no_recent_alerts:,} | "
                    f"failed: "
                    f"{n_failed:,}"
                )


            # ========================================================
            # If the target was reached in the middle of a seed,
            # stop here. Keep the checkpoint on the CURRENT seed.
            # If the target is increased later, rerunning the notebook
            # can revisit this seed and recover any unused remainder.
            # ========================================================

            if (
                len(existing_locus_ids)
                >= TARGET_TOTAL_LOCI
            ):
                break


            # ========================================================
            # Seed fully exhausted:
            # flush everything before advancing the checkpoint.
            # ========================================================

            if seed_exhausted:

                flush_alerts()
                flush_loci()

                seed_raw = (
                    n_raw_loci_seen
                    - seed_raw_start
                )

                seed_saved = (
                    n_loci_saved
                    - seed_saved_start
                )

                seed_duplicates = (
                    n_duplicate_loci
                    - seed_dup_start
                )

                seed_no_recent = (
                    n_no_recent_alerts
                    - seed_no_recent_start
                )

                seed_failed = (
                    n_failed
                    - seed_failed_start
                )

                print()
                print(
                    f"Seed {seed} exhausted | "
                    f"raw loci: {seed_raw:,} | "
                    f"new saved: {seed_saved:,} | "
                    f"duplicates: {seed_duplicates:,} | "
                    f"no recent alerts: {seed_no_recent:,} | "
                    f"failed: {seed_failed:,}"
                )

                # Advance checkpoint only AFTER the seed's data
                # have been safely flushed.
                save_seed_state(
                    seed + 1
                )

                seed += 1

            else:
                # Defensive fallback: if the iterator stopped for
                # some unexpected reason without reaching the target,
                # do not silently advance the seed.
                print(
                    f"Seed {seed} ended without being marked exhausted."
                )
                break


finally:

    # Always save completed buffered results when the cell exits,
    # including a normal stop or KeyboardInterrupt.
    flush_alerts()
    flush_loci()


print()
print("Finished")
print()

print(
    f"New unique loci saved this run:    "
    f"{n_loci_saved:,}"
)

print(
    f"New unique alerts saved this run:  "
    f"{n_alerts_saved:,}"
)

print(
    f"Duplicate loci skipped this run:   "
    f"{n_duplicate_loci:,}"
)

print(
    f"Duplicate alerts skipped this run: "
    f"{n_duplicate_alerts:,}"
)

print(
    f"No recent LSST alerts this run:    "
    f"{n_no_recent_alerts:,}"
)

print(
    f"Failed loci this run:              "
    f"{n_failed:,}"
)

print()

print(
    f"Total loci on disk:                "
    f"{len(existing_locus_ids):,}"
)

print(
    f"Target total loci:                 "
    f"{TARGET_TOTAL_LOCI:,}"
)

print(
    f"Total alerts on disk:              "
    f"{len(existing_alert_ids):,}"
)

print(
    f"Next/current seed checkpoint:      "
    f"{load_seed_state()}"
)


## 7. Quick validation

In [ ]:
loci_ds = ds.dataset(
    locus_dir,
    format="parquet",
)

alerts_ds = ds.dataset(
    alert_dir,
    format="parquet",
)


# ------------------------------------------------------------
# Loci
# ------------------------------------------------------------

locus_check = loci_ds.to_table(
    columns=[
        "locus_id",
        "num_tagged_alerts",
        "max_score",
        "milliquas_match",
    ]
)

locus_ids = locus_check[
    "locus_id"
].to_pylist()


# ------------------------------------------------------------
# Alerts
# ------------------------------------------------------------

alert_check = alerts_ds.to_table(
    columns=[
        "alert_id",
        "locus_id",
        "mjd",
    ]
)

alert_ids = alert_check[
    "alert_id"
].to_pylist()

alert_locus_ids = alert_check[
    "locus_id"
].to_pylist()

mjds = alert_check[
    "mjd"
].to_pylist()


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    f"Locus rows:       "
    f"{len(locus_ids):,}"
)

print(
    f"Unique locus IDs: "
    f"{len(set(locus_ids)):,}"
)

print(
    f"Target total:     "
    f"{TARGET_TOTAL_LOCI:,}"
)

print(
    f"Target reached:   "
    f"{len(set(locus_ids)) >= TARGET_TOTAL_LOCI}"
)

print()

print(
    f"Alert rows:       "
    f"{len(alert_ids):,}"
)

print(
    f"Unique alert IDs: "
    f"{len(set(alert_ids)):,}"
)

print()

if mjds:

    print(
        f"Earliest MJD:     "
        f"{min(mjds):.6f}"
    )

    print(
        f"All MJD >= cutoff:"
        f" {min(mjds) >= MJD_MIN}"
    )

else:

    print("No alerts found.")

print()

print(
    "All alert locus IDs exist in locus table:",
    set(alert_locus_ids).issubset(
        set(locus_ids)
    ),
)

print(
    "Milliquas-matched loci:",
    sum(
        x is True
        for x in locus_check[
            "milliquas_match"
        ].to_pylist()
    ),
)
